# Lab 4 - Malicious Software & File Integrity

## Aim

- Understand how file integrity monitoring works in practice.
- Build a simple integrity baseline using SHA-256 hashes.
- Detect modified, new, and missing files.
- Experiment with basic signature-based detection and a worm propagation simulation.


## Creating an Integrity Baseline

We walk a directory (`watched_folder/`), hash each file with SHA-256,
and store the results in `baseline.csv` as:

- file path
- SHA-256 hash
- timestamp (when the baseline was created)


In [1]:
import hashlib, csv, os, time

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def create_baseline(root_dir="watched_folder", out_file="baseline.csv"):
    rows = []
    for dirpath, _, filenames in os.walk(root_dir):
        for name in filenames:
            full = os.path.join(dirpath, name)
            rows.append({
                "file": full,
                "hash": sha256(full),
                "timestamp": time.time(),
            })
    with open(out_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "hash", "timestamp"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"Baseline written to {out_file}")

if __name__ == "__main__":
    create_baseline()


Baseline written to baseline.csv


## Checking for Changes

To detect tampering, we recompute hashes and compare with the baseline.

Files can be:

- **Modified** - existed before, but hash changed.
- **New** - not present in baseline.
- **Missing** - present in baseline, but file is gone.

This is the core idea behind host-based integrity monitoring tools.


In [2]:
import csv, hashlib, os

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def load_baseline(path="baseline.csv"):
    baseline = {}
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            baseline[row["file"]] = row["hash"]
    return baseline

def check(root_dir="watched_folder", baseline_file="baseline.csv"):
    baseline = load_baseline(baseline_file)
    current_files = {}
    for dirpath, _, filenames in os.walk(root_dir):
        for name in filenames:
            full = os.path.join(dirpath, name)
            current_files[full] = sha256(full)

    modified = []
    new_files = []
    missing = []

    for f, h in current_files.items():
        if f not in baseline:
            new_files.append(f)
        elif baseline[f] != h:
            modified.append(f)

    for f in baseline:
        if f not in current_files:
            missing.append(f)

    print("Modified files:", modified)
    print("New files:", new_files)
    print("Missing files:", missing)

if __name__ == "__main__":
    check()


Modified files: []
New files: []
Missing files: []


## Simple Signature-Based Scanner

We search for suspicious patterns in `.py` files, such as:

- `eval(` / `exec(` - dynamic execution of strings.
- `base64.b64decode` - sometimes used to hide payloads.
- `socket.socket` - could indicate networking behaviour.

Signature-based scanners match against known bad patterns or byte sequences.


In [3]:
import os
import re

SUSPICIOUS_PATTERNS = [
    re.compile(r"eval\("),
    re.compile(r"exec\("),
    re.compile(r"base64\.b64decode"),
    re.compile(r"socket\.socket"),
]

def scan_file(path):
    with open(path, "r", errors="ignore") as f:
        text = f.read()
    hits = []
    for pat in SUSPICIOUS_PATTERNS:
        if pat.search(text):
            hits.append(pat.pattern)
    return hits

def scan_dir(root_dir="."):
    for dirpath, _, filenames in os.walk(root_dir):
        for name in filenames:
            if name.endswith(".py"):
                full = os.path.join(dirpath, name)
                hits = scan_file(full)
                if hits:
                    print(f"[!] Suspicious patterns in {full}: {hits}")

if __name__ == "__main__":
    scan_dir(".")


## Worm Propagation Simulation

This toy model:

- Represents each host as a Boolean value (infected or not).
- Each infected host scans a number of random targets per step.
- Newly infected hosts become active in the next step.

The output shows how quickly a worm can spread with only basic scanning.


In [4]:
import random

def simulate_worm(num_hosts=50, initial_infected=1, scans_per_step=5, steps=10):
    hosts = [False] * num_hosts
    for i in range(initial_infected):
        hosts[i] = True

    for step in range(steps):
        newly_infected = []
        for i, infected in enumerate(hosts):
            if infected:
                for _ in range(scans_per_step):
                    target = random.randrange(num_hosts)
                    if not hosts[target]:
                        newly_infected.append(target)
        for idx in newly_infected:
            hosts[idx] = True
        print(f"Step {step}: {sum(hosts)} / {num_hosts} hosts infected")

if __name__ == "__main__":
    simulate_worm()


Step 0: 6 / 50 hosts infected
Step 1: 26 / 50 hosts infected
Step 2: 47 / 50 hosts infected
Step 3: 50 / 50 hosts infected
Step 4: 50 / 50 hosts infected
Step 5: 50 / 50 hosts infected
Step 6: 50 / 50 hosts infected
Step 7: 50 / 50 hosts infected
Step 8: 50 / 50 hosts infected
Step 9: 50 / 50 hosts infected
